In [3]:
%env AWS_PROFILE=platform-developer

env: AWS_PROFILE=platform-developer


In [ ]:
from utils.elasticsearch import get_client
from core.source import ElasticSource

pipeline_date = "2025-10-02"
index_date = "2025-10-02"
es_client = get_client("read_only", pipeline_date, "public")

source = ElasticSource(
    es_client=es_client,
    index_name=f"works-source-{index_date}",
    query={"match_all": {}}
)

source_works = {}

for work in source.stream_raw():
    source_id = work["state"]["sourceIdentifier"]
    if source_id["identifierType"]["id"] == "sierra-system-number":
        source_works[source_id["value"]] = work

2026-09-07 09:56:28 [info     ] Creating Elasticsearch client  es_mode=public host_config=https://0687c4947f404fe8ae10fa1aeab71bdc.eu-west-1.aws.found.io:443
2026-09-07 09:56:29 [info     ] Ran Elasticsearch query        duration_seconds=1 record_count=2000 slice_index=0
2026-09-07 09:56:30 [info     ] Ran Elasticsearch query        duration_seconds=2 record_count=2000 slice_index=3
2026-09-07 09:56:30 [info     ] Ran Elasticsearch query        duration_seconds=2 record_count=2000 slice_index=2
2026-09-07 09:56:30 [info     ] Ran Elasticsearch query        duration_seconds=3 record_count=2000 slice_index=4
2026-09-07 09:56:30 [info     ] Ran Elasticsearch query        duration_seconds=2 record_count=2000 slice_index=0
2026-09-07 09:56:31 [info     ] Ran Elasticsearch query        duration_seconds=3 record_count=2000 slice_index=1
2026-09-07 09:56:32 [info     ] Ran Elasticsearch query        duration_seconds=1 record_count=2000 slice_index=3
2026-09-07 09:56:33 [info     ] Ran Elastics

In [ ]:
import polars as pl
import pickle

SOURCE_WORKS_PATH = "data/sierra-source-works.parquet"

def save_parquet_snapshot(data: dict, path: str):
    df = pl.DataFrame({
        "id": list(data.keys()),
        "body": [pickle.dumps(v) for v in data.values()],
    })
    df.write_parquet(path)

save_parquet_snapshot(source_works, SOURCE_WORKS_PATH)